# Project 3 — Customer RFM Segmentation

Create customer-level Recency, Frequency, and Monetary measures, build interpretable customer segments, and recommend targeted actions. All customers and transactions are synthetic.

## Load customers and transactions

RFM uses customer purchase history. The customers table has one row per customer; the transactions table has one row per order. Check key uniqueness, transaction dates, and the relationship between tables before summarizing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path  # Resolve the data directory.
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")  # Support different notebook launch locations.
customers = pd.read_csv(DATA_DIR / "customers.csv", parse_dates=["signup_date"])  # Load customer attributes.
transactions = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["order_date"])  # Load order-level transactions.
print("Customers:", customers.shape, "| Transactions:", transactions.shape)  # Display both table sizes.
print("Unique customer IDs:", customers["customer_id"].nunique())  # Count unique customers.
print("Unique order IDs:", transactions["order_id"].nunique())  # Check the order identifier.
print("Transactions without a customer match:", (~transactions["customer_id"].isin(customers["customer_id"])).sum())  # Check referential integrity.
print(transactions.head())  # Preview transaction rows.

## Build Recency, Frequency, and Monetary

Recency is days since the customer's last order, frequency is distinct orders in the observation period, and monetary value is total transaction value. Fix the observation date so the analysis can be reproduced later.

In [ ]:
analysis_date = pd.Timestamp("2026-07-01")  # Fix the reference date for reproducibility.
rfm = transactions.groupby("customer_id").agg(  # Aggregate transactions at the customer level.
    last_order=("order_date", "max"),  # Find the last observed purchase.
    frequency=("order_id", "nunique"),  # Count distinct orders.
    monetary=("order_value", "sum"),  # Sum order value.
)  # Finish customer-level aggregation.
rfm["recency_days"] = (analysis_date - rfm["last_order"]).dt.days  # Calculate days since the last order.
rfm = rfm.reset_index().merge(customers, on="customer_id", how="left")  # Attach customer attributes to the RFM table.
print(rfm[["customer_id", "recency_days", "frequency", "monetary", "region", "acquisition_channel"]].head())  # Preview RFM features.

## Score customers and assign segments

Quartile scores make RFM measures easier to compare. Recent purchases should receive higher recency scores; higher frequency and monetary values should receive higher scores. Scores are relative to this dataset and observation window.

In [ ]:
rfm["R_score"] = pd.qcut(rfm["recency_days"].rank(method="first"), 4, labels=[4,3,2,1]).astype(int)  # Give more recent customers higher recency scores.
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)  # Give frequent customers higher frequency scores.
rfm["M_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)  # Give higher spenders higher monetary scores.
def assign_segment(row):  # Define a transparent, illustrative segment rule.
    if row["R_score"] >= 3 and row["F_score"] >= 3 and row["M_score"] >= 3:  # Identify recent, frequent, high-spend customers.
        return "Champions"  # Label the highest-value active group.
    if row["R_score"] <= 2 and row["F_score"] >= 3:  # Identify previously frequent customers who are not recent.
        return "At Risk"  # Label customers who may need re-engagement.
    if row["R_score"] >= 3 and row["F_score"] <= 2:  # Identify recent customers with lower repeat frequency.
        return "New / Promising"  # Label customers to nurture.
    if row["F_score"] >= 3:  # Catch other frequent purchasers.
        return "Loyal"  # Label loyal customers.
    return "Others"  # Place remaining customers into a broad residual group.
rfm["segment"] = rfm.apply(assign_segment, axis=1)  # Apply the rule to each customer.
print(rfm["segment"].value_counts())  # Count customers by segment.
print(rfm.groupby("segment").agg(customers=("customer_id","nunique"), revenue=("monetary","sum"), avg_frequency=("frequency","mean"), median_recency=("recency_days","median")).round(2))  # Compare segment size and value.

## Visualize segments and priorities

A segment table shows size and value; a chart helps stakeholders see the trade-off between a segment's revenue contribution and how many customers it contains. Investigate whether segment strategies are operationally feasible.

In [ ]:
segment_report = rfm.groupby("segment").agg(  # Summarize the customer segments.
    customers=("customer_id", "nunique"),  # Count customers in each segment.
    revenue=("monetary", "sum"),  # Sum transaction value by segment.
    avg_frequency=("frequency", "mean"),  # Calculate average distinct orders per customer.
)  # Finish segment report.
segment_report["revenue_share"] = segment_report["revenue"] / segment_report["revenue"].sum()  # Calculate segment share of revenue.
fig, ax = plt.subplots(figsize=(8, 4))  # Create a figure for the segment comparison.
segment_report.sort_values("revenue", ascending=False)["revenue"].plot(kind="bar", ax=ax)  # Plot revenue by segment.
ax.set_title("Revenue by RFM Segment")  # Add a chart title.
ax.set_xlabel("RFM Segment")  # Label the category axis.
ax.set_ylabel("Revenue (synthetic currency units)")  # Label the value axis.
plt.xticks(rotation=25, ha="right")  # Rotate labels so they remain legible.
plt.tight_layout()  # Adjust layout.
plt.show()  # Display the chart.
print(segment_report.round(3))  # Show the values behind the chart.

## Make a customer strategy recommendation

Use segment metrics to propose differentiated strategies. For example, nurture promising recent customers, test a win-back campaign for at-risk customers, and protect the experience of high-value customers. Measure incremental outcomes with a control group where feasible.

In [ ]:
strategy = {  # Map segments to testable business actions.
    "Champions": "Protect service quality and test a loyalty benefit.",  # Preserve value from engaged, high-value buyers.
    "At Risk": "Test a targeted win-back message against a holdout group.",  # Check whether outreach changes behavior.
    "New / Promising": "Test onboarding and second-purchase reminders.",  # Encourage a second purchase.
    "Loyal": "Offer relevant bundles and monitor contribution margin.",  # Increase value without over-discounting.
    "Others": "Use low-cost experiments before expensive personalization.",  # Avoid overspending on broad residual groups.
}  # Finish the strategy map.
for segment, action in strategy.items():  # Walk through each segment strategy.
    print(f"{segment}: {action}")  # Display a recommendation to test.
print("Caution: RFM labels are heuristic; validate campaign effects with experiments and margin data.")  # State an important limitation.